# Problem 07 | Student Placement Eligibility

**Objective:** Build a binary classification solution to predict student placement eligibility.

**Algorithm:** Logistic Regression  
**Evaluation:** Accuracy, Precision, Recall, F1-score, ROC-AUC and Confusion Matrix.

> This notebook follows the provided Learn Depth requirements and uses the supplied `dataset_07_student_placement_eligibility.csv`.


## 1. Import Libraries and Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)

df = pd.read_csv("dataset_07_student_placement_eligibility.csv")
print("Shape:", df.shape)
display(df.head())


## 2. Dataset Inspection

The dataset contains six predictor variables and one binary target. The target is `1` for the positive/event class and `0` for the negative/non-event class.


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDescriptive statistics:")
display(df.describe().T)


## 3. Data Quality Checks

Check missing values, duplicate rows, class balance, and basic validity ranges.


In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing"))

print("Duplicate rows:", df.duplicated().sum())

print("\nTarget balance:")
display(df["target"].value_counts().sort_index().to_frame("count"))

print("\nTarget proportions:")
display((df["target"].value_counts(normalize=True).sort_index() * 100).round(2).to_frame("percent"))

print("\nObserved ranges:")
display(df.agg(["min", "max"]).T)


## 4. Outlier Check

IQR-based checks are used to flag unusually low/high observations. An IQR flag does **not** automatically mean the value is wrong; plausible observations are retained unless there is evidence of invalidity.


In [ ]:
numeric_features = df.drop(columns="target").columns

outlier_summary = []
for col in numeric_features:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append([col, lower, upper, int(count)])

outlier_summary = pd.DataFrame(
    outlier_summary, columns=["feature", "lower_bound", "upper_bound", "outlier_count"]
)
display(outlier_summary)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.ravel(), numeric_features):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(f"Boxplot: {col}")
plt.tight_layout()
plt.show()


## 5. Exploratory Analysis

Compare feature distributions and their relationship with the target.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.ravel(), numeric_features):
    sns.histplot(data=df, x=col, hue="target", kde=True, element="step", ax=ax)
    ax.set_title(f"{col} by target")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## 6. Preprocessing and Train/Test Split

There are no missing values or duplicate rows in the supplied dataset, so imputation is not required. Numeric predictors are standardized because Logistic Regression is sensitive to feature scale. The target is separated before modeling.

A stratified 80/20 train-test split is used with `random_state=42` so the class ratio is preserved and the experiment is reproducible.


In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        solver="lbfgs",
        penalty="l2",
        C=1.0,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


## 7. Logistic Regression Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Score": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob)
    ]
})
metrics["Score"] = metrics["Score"].round(4)
display(metrics)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicted 0", "Predicted 1"],
    yticklabels=["Actual 0", "Actual 1"]
)
plt.title("Confusion Matrix")
plt.xlabel("Prediction")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

print("TN, FP, FN, TP =", cm.ravel())


## 9. ROC Curve and AUC

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve - Logistic Regression")
plt.tight_layout()
plt.show()


## 10. Coefficient / Feature Effect Interpretation

The model uses standardized predictors. A positive coefficient increases the log-odds of the positive class (`target=1`), while a negative coefficient decreases it, holding other variables constant.

Coefficient magnitude can be used to compare the direction and relative strength of the modeled effects, but it should not be interpreted as proof of causation.


In [ ]:
coefficients = pd.Series(
    model.named_steps["logistic_regression"].coef_[0],
    index=X.columns
).sort_values()

display(coefficients.to_frame("standardized_coefficient"))

plt.figure(figsize=(9, 5))
coefficients.plot(kind="barh")
plt.axvline(0, linewidth=1)
plt.title("Logistic Regression Coefficients")
plt.xlabel("Standardized coefficient")
plt.tight_layout()
plt.show()


## 11. Practical Interpretation and Limitations

### Key observations from this run
- The classes are balanced in the supplied dataset.
- No missing values or duplicate rows were found.
- The observed feature ranges do not show obvious invalid values based on the supplied columns.
- IQR analysis flags some observations as statistical outliers; these were not automatically removed because an outlier is not necessarily an error.
- Standardization was applied before Logistic Regression.
- The final evaluation uses a held-out test set.

### Limitations
- The dataset is limited to the provided predictors and may not represent all real placement factors.
- Logistic Regression assumes a linear relationship between predictors and the log-odds of the target.
- Performance on this dataset may not generalize to another college, cohort, or recruitment process.
- A larger external validation set would provide stronger evidence of generalization.
- Threshold tuning, cross-validation, regularization comparison, and alternative models could be explored as future improvements.


## 12. Conclusion

The project demonstrates a reproducible binary classification workflow for Student Placement Eligibility: dataset inspection, data-quality checks, preprocessing, stratified train/test splitting, Logistic Regression, metric-based evaluation, confusion-matrix analysis, ROC-AUC assessment, and coefficient interpretation.
